In [ ]:
# --- Colab Notebook Example ----------------------------------------------
# Cell 1: !pip install sklearn pandas mlxtend
# Cell 2: from google.colab import drive; drive.mount('/content/drive')
# Cell 3: from google.colab import files; files.upload()
# Cell 4: import task3_integration as tri
#          cf,hybrid,df=tri.main('/content/train.csv','/content/test.csv')
# Cell 5: uid=df['User_id'].iloc[0]; hist=df[df['User_id']==uid]['itemDescription'].tolist()
#          miner=tri.FrequentPatternMiner().fit(df,user_id=uid)
#          pats=miner.get_patterns(5)
#          print(tri.CollaborativeFilteringRecommender().recommend(uid,exclude_items=hist))
#          print(hybrid.recommend(uid,pats,exclude=hist))
# Cell 6: pc,ph=tri.evaluate_models(cf,hybrid,df,pd.read_csv('/content/test.csv')); print(pc,ph)

In [32]:
!pip install sklearn
!pip install pandas
!pip install mlxtend

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [33]:
from google.colab import drive; drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
from google.colab import files; files.upload()

Saving task3_final.py to task3_final.py


{'task3_final.py': b'# -*- coding: utf-8 -*-\n"""Untitled9.ipynb\n\nAutomatically generated by Colab.\n\nOriginal file is located at\n    https://colab.research.google.com/drive/1ZRV4B3zLxsw9WxsvDmA7y4ia90Pm-FVB\n"""\n\n# task3_integration.py\n"""\nTask 3: System Integration & Interface for Grocery Recommender\n\nThis module is self-contained (no external Task1/Task2 modules needed).\nIt integrates:\n  - AprioriMiner & FrequentPatternMiner (Task 1)\n  - CollaborativeFilteringRecommender (Task 2)\n  - HybridRecommender & Evaluation (Task 3)\n\nUsage:\n As script:\n   python task3_integration.py train.csv [test.csv]\n In Colab/notebook:\n   cf, hybrid, train_df = main(train_path, test_path)\n"""\nimport pandas as pd\nimport numpy as np\nimport os, sys\nfrom collections import defaultdict\nfrom itertools import combinations\nfrom sklearn.metrics.pairwise import cosine_similarity\n\n# --- Task 1: AprioriMiner & FrequentPatternMiner -------------------------\nclass AprioriMiner:\n    """Imp

In [35]:
import task3_final as tri
cf,hybrid,df=tri.main('/content/drive/MyDrive/big data mining/assignment 3/Groceries data train.csv','/content/drive/MyDrive/big data mining/assignment 3/Groceries data test.csv')

In [36]:
uid=df['User_id'].iloc[0]; hist=df[df['User_id']==uid]['itemDescription'].tolist()
miner=tri.FrequentPatternMiner().fit(df,user_id=uid)
pats=miner.get_patterns(5)
print(tri.CollaborativeFilteringRecommender().recommend(uid,exclude_items=hist))
print(hybrid.recommend(uid,pats,exclude=hist))

[]
['whole milk', 'rolls/buns', 'other vegetables', 'root vegetables', 'newspapers']


In [39]:
print(pats)

[(('shopping bags',), 0.5), (('cleaner',), 0.5), (('instant coffee',), 0.5), (('domestic eggs',), 0.5), (('shopping bags', 'cleaner'), 0.5)]


In [37]:
pc,ph=tri.evaluate_models(cf,hybrid,df,pd.read_csv('/content/drive/MyDrive/big data mining/assignment 3/Groceries data test.csv')); print(pc,ph)

KeyError: 'User_id'

In [38]:
# Cell: Test the integrated module with a tiny toy dataset

import pandas as pd
from task3_final import (
    FrequentPatternMiner,
    CollaborativeFilteringRecommender,
    HybridRecommender,
    evaluate_models
)

# 1) Build a tiny dataset
data = pd.DataFrame({
    'User_id':       ['U1','U1','U1','U2','U2','U3','U3'],
    'Date':          ['2025-04-01','2025-04-02','2025-04-05',
                      '2025-04-01','2025-04-03',
                      '2025-04-02','2025-04-04'],
    'itemDescription':['apple','banana','carrot',
                      'banana','apple',
                      'banana','date']
})

# 2) Task 1: Mine patterns for U1
miner = FrequentPatternMiner(min_support=0.3).fit(data, user_id='U1')
patterns = miner.get_patterns(top_n=5)
print("Top patterns for U1:", patterns)

# 3) Task 2: Fit CF and get recommendations for U1
cf = CollaborativeFilteringRecommender().fit(data)
history_U1 = data[data['User_id']=='U1']['itemDescription'].tolist()
cf_recs = cf.recommend('U1', exclude_items=history_U1)
print("CF-only recs for U1:", cf_recs)

# 4) Task 3: Blend into hybrid recommendations
hybrid = HybridRecommender(cf_model=cf, alpha=0.5)
hybr_recs = hybrid.recommend('U1', patterns=patterns, exclude=history_U1)
print("Hybrid recs for U1:", hybr_recs)

# 5) (Optional) Evaluate on a “test set” that’s just held‑out rows
test = pd.DataFrame({
    'User_id':       ['U1','U2','U3'],
    'itemDescription':['date','carrot','apple']
})
p_cf, p_h = evaluate_models(cf, hybrid, data, test, top_n=3)
print(f"Precision@3 — CF-only: {p_cf:.2f}, Hybrid: {p_h:.2f}")


Top patterns for U1: [(('apple',), 0.3333333333333333), (('banana',), 0.3333333333333333), (('carrot',), 0.3333333333333333)]
CF-only recs for U1: ['date']
Hybrid recs for U1: ['date']
Precision@3 — CF-only: 0.33, Hybrid: 0.33
